<a href="https://colab.research.google.com/github/shivakamibaiju-alt/ml_lab/blob/main/Experiment12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic Survival Prediction: Bagging vs AdaBoost

This notebook trains and evaluates Bagging and AdaBoost classifiers on the Titanic dataset.

**Dataset required:** `Titanic.csv` in the same folder as this notebook. The file must contain the columns used below, including `Survived`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

## 1. Load the Titanic dataset

In [3]:
import os
import pandas as pd

# Try to load the local file, or fall back to a public URL if it does not exist
file_path = "Titanic.csv"
if not os.path.exists(file_path):
    print(f"'{file_path}' not found locally. Loading standard Titanic dataset from GitHub...")
    file_path = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(file_path)
df.head()

'Titanic.csv' not found locally. Loading standard Titanic dataset from GitHub...


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Select features and target

In [6]:
features = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]
X = df[features]
y = df["Survived"]

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

## 3. Preprocess the features

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Ensure feature lists are defined
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

# Create a preprocessor that imputes missing values and encodes categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

## 4. Split the dataset

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 712
Testing samples: 179


## 5. Train the Bagging classifier

In [9]:
base_model = DecisionTreeClassifier(max_depth=4, random_state=42)

bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    random_state=42
)

bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)

## 6. Train the AdaBoost classifier

In [10]:
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=50,
    random_state=42
)

boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

## 7. Evaluate the models

In [11]:
def evaluate_model(name, y_true, y_pred):
    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))

evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)


 Bagging
Accuracy: 0.8045
Precision: 0.8696
Recall: 0.5797
F1-score: 0.6957

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774


## Conclusion

Compare the accuracy, precision, recall, and F1-score for both models. The better model depends on the evaluation results and the goal of the prediction.